In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

dataset_path = "/content/drive/MyDrive/ADSProject/IBM_Dataset"

print(os.listdir(dataset_path))

['Location.xlsx', 'Services.xlsx', 'Demographics.xlsx', 'Status.xlsx', 'merged_df.xlsx', 'Population.xlsx']


Reading Datasets

In [4]:
!pip install openpyxl

In [5]:
import pandas as pd

# Demographics Dataset
demographics_df = pd.read_excel(
    "/content/drive/MyDrive/ADSProject/IBM_Dataset/Demographics.xlsx"
)

# Location Dataset
location_df = pd.read_excel(
    "/content/drive/MyDrive/ADSProject/IBM_Dataset/Location.xlsx"
)

# Population Dataset
population_df = pd.read_excel(
    "/content/drive/MyDrive/ADSProject/IBM_Dataset/Population.xlsx"
)

# Services Dataset
services_df = pd.read_excel(
    "/content/drive/MyDrive/ADSProject/IBM_Dataset/Services.xlsx"
)

# Statud Dataset
status_df = pd.read_excel(
    "/content/drive/MyDrive/ADSProject/IBM_Dataset/Status.xlsx"
)

In [6]:
# Row count of each dataset

print("Demographics:", demographics_df.shape)
print("Services:", services_df.shape)
print("Status:", status_df.shape)
print("Location:", location_df.shape)
print("Population:", population_df.shape)

# Unique CustomerIDs
print("\nNumber of Unique Customer IDs")
print(demographics_df["Customer ID"].nunique())
print(services_df["Customer ID"].nunique())
print(status_df["Customer ID"].nunique())
print(location_df["Customer ID"].nunique())
# print(population_df["Customer ID"].nunique())

Demographics: (7043, 9)
Services: (7043, 31)
Status: (7043, 11)
Location: (7043, 10)
Population: (1671, 3)

Number of Unique Customer IDs
7043
7043
7043
7043


Merge Datasets

In [7]:
master_df = demographics_df.merge(
    services_df,
    on="Customer ID",
    how="inner"
)

master_df = master_df.merge(
    status_df,
    on="Customer ID",
    how="inner"
)

# Same column exists in master_df and location_df, thus error
# master_df = master_df.merge(
#     location_df,
#     on="Customer ID",
#     how="inner"
# )

In [8]:
# Columns of both datasets

print("Location columns:")
print(location_df.columns.tolist())

print("\nMaster columns:")
print(master_df.columns.tolist())

Location columns:
['Location ID', 'Customer ID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude']

Master columns:
['Customer ID', 'Count_x', 'Gender', 'Age', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Number of Dependents', 'Service ID', 'Count_y', 'Quarter_x', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'Status ID', 'Count', 'Quarter_y', 'Satisfaction Score', 'Customer Status', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reas

In [9]:
# Check what value the 'Count' column helds

print(master_df["Count"].value_counts())
print(location_df["Count"].value_counts())

Count
1    7043
Name: count, dtype: int64
Count
1    7043
Name: count, dtype: int64


In [10]:
# Removing 'Count' columns, as it doesnt contribute to prediction

master_df = master_df.drop(columns=["Count", "Count_x", "Count_y"])
location_df = location_df.drop(columns=["Count"])

In [11]:
# Checking if there are still any other common columns are present or not

common_cols = set(master_df.columns).intersection(location_df.columns)
print(common_cols)

{'Customer ID'}


In [12]:
# Since no common columns other than 'Customer ID', proceeding to merge
master_df = master_df.merge(
    location_df,
    on="Customer ID",
    how="inner"
)

In [13]:
print(master_df.shape)

(7043, 54)


Merging 'Population' Dataset

In [14]:

print(location_df.columns.tolist())
print(population_df.columns.tolist())

['Location ID', 'Customer ID', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude']
['ID', 'Zip Code', 'Population']


In [15]:
# Merge 'Population' Dataset
# Why left join instead of inner?

# master_df has 7043 customers.
# population_df has 1671 ZIP codes.

# Many customers share the same ZIP code, so 1671 ZIP codes are enough to cover 7043 customers.

# Using a left join ensures:

# Every customer stays in the dataset.
# Each customer gets the population of their ZIP code.
# If a ZIP code is missing from population_df, only the Population value becomes NaN; the customer is not removed.

master_df = master_df.merge(
    population_df,
    on="Zip Code",
    how="left"
)

In [16]:
print(master_df.shape)
print(master_df["Population"].isnull().sum())

(7043, 56)
0


In [17]:
import os

# Define ytarget folder and filename
folder_path = '/content/drive/MyDrive/ADSProject/Project_Root/Merged_Dataset'
file_name = 'master_df.xlsx'
full_path = os.path.join(folder_path, file_name)

# Save the DataFrame to Google Drive
master_df.to_excel(full_path, index=False)

In [18]:
import os
print("File exists:", os.path.exists(full_path))

File exists: True
